# Lab 2.2 &mdash; ReAct, and the Parser You Do Not Have to Write

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Implement the classic text ReAct format &mdash; Thought / Action / Observation
- Collect the eight ways a model drifts out of that format
- Discover the failure a parser cannot catch: a valid format with an unusable argument
- Do the same job with <code>bind_tools</code>, where the argument is schema-checked

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 2.1.** Same case file. This lab is the strongest argument in Module 2
> for using the framework rather than reimplementing it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 2 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the two tools, carried through Module 2
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1003'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    """
    rec = LEDGER.get(ref)
    return json.dumps({"ref": ref, **rec}) if rec else f"no payment found with reference {ref!r}"


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {t.name: t for t in (lookup_payment, policy_for)}
print("tools:", list(TOOLS))

## Concept

**ReAct** interleaves reasoning and acting: *Thought* (what do I know?), *Action* (what shall I
do?), *Observation* (what came back?), repeat. That is the loop from Module 1, given a shape.

The original formulation asks the model to emit that shape **as text**, which means you must parse
it. Every parser is a contract, and the model has not signed it. This lab makes you write that
parser, breaks it, and then shows you the version where the contract is enforced by the API
instead of by your regex.

## Section 1 &mdash; The text format, and the parser it needs

The happy path looks like this:

```
Thought: I need the payment record first.
Action: lookup_payment
Action Input: PMT-1003
```

Pull the three fields out. Return `None` for a step that does not carry an action &mdash; that is how
a final answer looks.

In [ ]:
import re

STEP_RE = re.compile(
    r"Thought:\s*(?P<thought>.*?)\s*"
    r"Action:\s*(?P<action>[\w_]+)\s*"
    r"Action Input:\s*(?P<input>.*?)\s*$",
    re.DOTALL | re.IGNORECASE)

def parse_step(text: str) -> dict | None:
    """Return {"thought", "action", "input"} or None if this is not an action step."""
    m = STEP_RE.search(text or "")
    if not m:
        return None
    return {"thought": m.group("thought").strip(),
            "action": m.group("action").strip(),
            "input": m.group("input").strip().strip('"').strip("'")}

In [ ]:
# --- Self-check: Section 1   (pure string work -- no model call)
_good = "Thought: I need the record.\nAction: lookup_payment\nAction Input: PMT-1003"

check("a well-formed step parses",       lambda: parse_step(_good) is not None)
check("the tool name is extracted",      lambda: parse_step(_good)["action"] == "lookup_payment")
check("the input is extracted",          lambda: parse_step(_good)["input"] == "PMT-1003")
check("the thought is extracted",        lambda: "record" in parse_step(_good)["thought"])
check("a final answer is not an action", lambda: parse_step("The payment needs Treasury.") is None)
check("quotes around the input are stripped",
      lambda: parse_step('Thought: t\nAction: lookup_payment\nAction Input: "PMT-1003"')["input"]
              == "PMT-1003")

## Section 2 &mdash; Eight ways it drifts

None of these is a badly behaved model. Every one is a reasonable thing for a fluent writer to
produce, and every one breaks a parser that was written against the happy path.

Decide which your parser should accept. There is no free answer here: a **lenient** parser
guesses and is sometimes wrong; a **strict** one refuses and costs you a retry.

In [ ]:
DRIFT = [
    ("markdown fences",   "```\nThought: t\nAction: lookup_payment\nAction Input: PMT-1003\n```"),
    ("bold headings",     "**Thought:** t\n**Action:** lookup_payment\n**Action Input:** PMT-1003"),
    ("numbered steps",    "1. Thought: t\n2. Action: lookup_payment\n3. Action Input: PMT-1003"),
    ("json instead",      '{"thought": "t", "action": "lookup_payment", "input": "PMT-1003"}'),
    ("prose action",      "Thought: t\nAction: I will call lookup_payment\nAction Input: PMT-1003"),
    ("missing input",     "Thought: t\nAction: lookup_payment"),
    ("two actions",       "Thought: t\nAction: lookup_payment\nAction Input: PMT-1003\n"
                          "Action: policy_for\nAction Input: LIMIT_BREACH"),
    ("preamble",          "Certainly! Here is my reasoning.\n\nThought: t\n"
                          "Action: lookup_payment\nAction Input: PMT-1003"),
]

def survives(text: str) -> bool:
    """Does the Section 1 parser get a usable tool name out of this?"""
    step = parse_step(text)
    return bool(step) and step["action"] in TOOLS

def _drift_table():
    for label, text in DRIFT:
        print(f"  [{'ok  ' if survives(text) else 'LOST'}] {label}")
guard(_drift_table)

In [ ]:
# --- Self-check: Section 2   (what a parser must and must not do)
_by = dict(DRIFT)

check("the plain happy path still works",
      lambda: survives("Thought: t\nAction: lookup_payment\nAction Input: PMT-1003"))
check("a prose action does NOT yield a real tool",
      lambda: survives(_by["prose action"]) is False,
      '"I will call lookup_payment" must not be accepted as the tool name')
check("a missing input is not silently accepted",
      lambda: parse_step(_by["missing input"]) is None,
      "half a step is not a step -- accepting it invents an argument")
check("raw JSON is not the text format",
      lambda: parse_step(_by["json instead"]) is None,
      "this one is worth noting: the model produced something BETTER, and the parser rejects it")
check("at least three of the eight drift cases are lost",
      lambda: sum(1 for _, t in DRIFT if not survives(t)) >= 3,
      "if your parser accepts everything it is guessing, which is worse")

## Section 3 &mdash; The same job, with no parser at all

`llm.bind_tools([...])` gives the model a tool **schema** rather than a format instruction. What
comes back is `AIMessage.tool_calls` &mdash; a list of `{"name", "args", "id"}` produced by the
serving stack, not by the model's prose. There is no format to drift out of, because there is no
format: the name is validated against the tools you passed, and the arguments against their
schema.

Write the step that turns one of those calls into a result.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

REACT_SYSTEM = ("You investigate payment exceptions. Use the tools to find the payment's reason "
                "code and the policy for it, then answer with the single next action.")

def native_step(model, messages: list) -> tuple[AIMessage, list]:
    """One turn: ask the model, run whatever it asked for, return (reply, tool results)."""
    ai = model.invoke(messages)
    results = []
    for call in ai.tool_calls:
        tool_obj = TOOLS.get(call["name"])
        content = tool_obj.invoke(call["args"]) if tool_obj else f"no such tool {call['name']!r}"
        results.append(ToolMessage(content=str(content), tool_call_id=call["id"]))
    return ai, results


def native_loop(question: str, max_steps: int = 6) -> dict:
    """The ReAct loop with no parser in it."""
    model = get_llm().bind_tools(list(TOOLS.values()))
    messages = [SystemMessage(REACT_SYSTEM), HumanMessage(question)]
    for step in range(max_steps):
        ai, results = native_step(model, messages)
        messages.append(ai)
        if not ai.tool_calls:
            return {"messages": messages, "steps": step, "answer": ai.content}
        messages.extend(results)
    return {"messages": messages, "steps": max_steps, "answer": "(step budget spent)"}

In [ ]:
# --- Self-check: Section 3   (a scripted model -- real message objects, no endpoint)
class _ScriptedModel:
    """Stands in for a bound model so the loop can be tested without the gateway."""
    def __init__(self, replies): self._replies = list(replies)
    def invoke(self, messages): return self._replies.pop(0)

def _call(name, args, cid):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args,
                                              "id": cid, "type": "tool_call"}])

def _scripted():
    return _ScriptedModel([
        _call("lookup_payment", {"ref": "PMT-1003"}, "c1"),
        _call("policy_for", {"reason_code": "LIMIT_BREACH"}, "c2"),
        AIMessage("Escalate to Treasury for approval."),
    ])

check("a tool call produces one ToolMessage",
      lambda: len(native_step(_scripted(), [HumanMessage("q")])[1]) == 1)
check("the ToolMessage carries the call's id",
      lambda: native_step(_scripted(), [HumanMessage("q")])[1][0].tool_call_id == "c1",
      "tool_call_id must be call['id'] -- that is what pairs result to request")
check("the tool actually ran",
      lambda: "LIMIT_BREACH" in native_step(_scripted(), [HumanMessage("q")])[1][0].content)
check("an unknown tool is reported, not raised",
      lambda: "no such tool" in native_step(
          _ScriptedModel([_call("delete_everything", {}, "c1")]), [HumanMessage("q")])[1][0].content,
      "the model can only name tools you gave it, but defend the boundary anyway")
check("there is no parser in this path at all",
      lambda: "parse_step" not in native_step.__code__.co_names,
      "that is the entire point of the section")

## Section 4 &mdash; ...and the version you would actually ship

`create_agent` is `native_loop` with the budget, the dispatch, the error handling and the message
bookkeeping already written. You have now built it twice, so you know exactly what it is doing.

In [ ]:
from langchain.agents import create_agent

def built_agent():
    """The same ReAct behaviour, as one call."""
    return create_agent(model=get_llm(), tools=list(TOOLS.values()),
                        system_prompt=REACT_SYSTEM)

## Run it for real

Three paths, one question. The first asks the model to produce the text format and parses it; the
second and third let the serving stack carry the structure.

In [ ]:
if llm_ready():
    STRICT = ("You investigate payment exceptions. Reply in EXACTLY this format and nothing "
              "else:\n"
              "Thought: <your reasoning>\n"
              "Action: <one of: " + ", ".join(TOOLS) + ">\n"
              "Action Input: <the argument, a bare value with no quotes or braces>")

    # The same instruction after six months of well-meaning edits. Nobody writes the loose one
    # on purpose; prompts drift towards "be helpful and thorough" one review at a time.
    LOOSE  = ("You investigate payment exceptions. Think step by step, explain your reasoning "
              "clearly for the operations team, and use the Thought / Action / Action Input "
              "format to call one of these tools: " + ", ".join(TOOLS) + ". Be thorough.")

    def _text_path():
        attempts = 4
        for label, instruction in (("strict", STRICT), ("loose", LOOSE)):
            parsed = usable = 0
            first = None
            for i in range(attempts):
                reply = ask("Why is PMT-1003 held, and what must we do about it?",
                            system=instruction)
                step = parse_step(reply)
                ok = bool(step) and step["action"] in TOOLS
                parsed += ok
                good = False
                if ok:
                    arg = list(TOOLS[step["action"]].args)[0]
                    result = TOOLS[step["action"]].invoke({arg: step["input"]})
                    good = "no payment found" not in str(result)
                usable += good
                if i == 0:
                    first = (reply, step, good)
            print(f"=== {label} instruction: {parsed}/{attempts} parsed, "
                  f"{usable}/{attempts} usable ===")
            reply, step, good = first
            print("  Action Input as written:",
                  repr(step["input"]) if step else "(did not parse)")
            print(f"  the tool could use it : {good}\n")
        return None
    TEXT_RESULT = guard(_text_path)

In [ ]:
if llm_ready():
    def _native_path():
        out = native_loop("Why is PMT-1003 held, and what must we do about it?")
        show_messages(out["messages"])
        print(f"\nsteps: {out['steps']}   answer: {out['answer'][:160]}")
        print("tool calls parsed by hand: 0")
    guard(_native_path)

In [ ]:
if llm_ready():
    def _built():
        out = built_agent().invoke(
            {"messages": [("human", "Why is PMT-1005 held, and what must we do about it?")]})
        show_messages(out["messages"])
    guard(_built)

### Read the trace

The result here is probably not the one you expected, and it is better than the one you expected.

**The text path parsed fine.** Given an explicit format instruction, the model very likely obeyed
every time. So the lesson is *not* "models cannot follow a format" &mdash; on a good day they can.

**Look at the second number.** `parsed` counts replies where the regex found a tool name.
`usable` counts replies where the extracted argument actually worked when passed to the tool. If
those two numbers differ, read the `Action Input` in the raw reply: models frequently write

```
Action Input: {"payment_id": "PMT-1003"}
```

instead of `PMT-1003`. The format is perfect. The tool name is right. The regex is delighted. And
the argument is a JSON object with a key your tool has never heard of &mdash; so the lookup returns
nothing, the agent concludes the payment does not exist, and **nothing anywhere reports an
error**. Your parser cannot catch this, because validating the argument was never its job.

That is the real cost of the text format, and it is worse than a parse failure. A parse failure is
loud. This is silent.

**The native path cannot fail that way.** `bind_tools` sends the tool's argument *schema*, and the
serving stack validates against it before your code sees anything. A wrong key is rejected at the
boundary. What is left to go wrong &mdash; wrong tool, wrong value &mdash; are real mistakes you can
see and measure.

**The built agent** is the native path with the budget and the bookkeeping done.

Keep the parser in your notes for the day you meet a model with no tool-calling API. That is the
only situation in which you should write one, and now you know exactly what you would be taking
on: not eight formatting edge cases, but every argument your tools accept, forever.

In [ ]:
score()

## Your turn

1. Make `parse_step` lenient enough to survive the markdown-fence and bold-heading cases. Then
   feed it `"Action: I will call lookup_payment"` and decide whether your leniency has started
   guessing.
2. The text path retries the same prompt six times. Add a **repair** turn instead: when parsing
   fails, send the model its own bad output and ask it to restate. Count the extra calls. That is
   the real cost of the text format.
3. Give `built_agent` a `response_format` (Lab 1.3) so it returns a typed action rather than a
   sentence, and check whether the answer to PMT-1005 still says Compliance.